# 공개용 작업 증빙 사본
[확실] 기존 파일의 코드·설명을 복사하고 출력·실행 번호·첨부 이미지를 제거했습니다. 개인 경로는 `private_inputs`로 바꿨습니다. 원자료 및 상세 결과는 포함하지 않습니다.
[확실] 이 사본을 새로 전체 실행한 기록은 아닙니다. 과거 실행 범위는 `evidence/verification.json`과 `docs/VALIDATION.md`를 확인하세요.


# 월·연령·점수별 활동 저조 공간 지도
[확실] 기존 분석 결과 CSV를 읽어 지도 대시보드 HTML을 생성합니다. 모델·선정 결과를 재계산하지 않습니다.
**VS Code: Python 커널 선택 → 모두 실행 → 마지막에 출력되는 HTML 경로 열기.**
지도의 월·연령·지역·점수 조건은 브라우저에서 바로 바꿀 수 있습니다.

[확실] ACTIVITY_RESIDUAL은 작을수록 유사 환경 대비 활동이 저조합니다.
[추정] 기본 색상은 활동저조강도=max(0,-ACTIVITY_RESIDUAL)를 사용하며 전체 자료의 양수 강도 99백분위를 공통 상한으로 둡니다. 상한 이상의 격자도 유지하되 색만 포화됩니다. 원점수·강도는 클릭 시 표시합니다.
[확실] 상대순위 모드는 지역×월×연령별 순위 백분위입니다. 월 간 절대적인 악화·개선을 뜻하지 않습니다.
[추정] 부드러운 히트맵은 주변 점들의 가중치를 합친 시각적 표현으로 점의 밀도·확대 수준에도 영향을 받습니다. 점수 자체의 공간 비교에는 기본 50m 격자 모드를 사용하세요.
[확실] 공간 선별 결과이며 사람의 고립 확정 위치가 아닙니다. 배경지도·Leaflet 라이브러리 로딩에는 인터넷이 필요합니다.

In [ ]:
%pip install -q pandas numpy pyproj

In [ ]:
from pathlib import Path
from datetime import datetime
import json,html,webbrowser
import numpy as np
import pandas as pd
from pyproj import Transformer
from IPython.display import display, HTML
SOURCE=Path(r'./private_inputs/main_work\monthly_grid_results\20261025_195726_925731')
OUTPUT=SOURCE/'visualizations'/datetime.now().strftime('%Y%m%d_%H%M%S')
OPEN_BROWSER=True
OUTPUT.mkdir(parents=True,exist_ok=True)
print('입력:',SOURCE)
print('지도 저장 위치:',OUTPUT)

## 1. 필요한 열만 읽고 좌표·그룹 검증
[확실] 320MB 전체 CSV에서 지도에 필요한 열만 읽습니다. 좌표는 EPSG:5179에서 WGS84(EPSG:4326)로 변환합니다. 잘못된 좌표·점수·중복을 조용히 버리지 않고 오류로 표시합니다.

In [ ]:
cols=['STD_YM','REGION','AGE','GRID_ID','DONG','X_COORD','Y_COORD','ACTIVITY_RESIDUAL','RANK','SELECTED','FLOW','PEER_MEDIAN','PEER_DISTANCE_REVIEW']
df=pd.read_csv(SOURCE/'04_all_scores.csv',usecols=cols,dtype={'STD_YM':str,'GRID_ID':str,'AGE':str})
for c in ['SELECTED','PEER_DISTANCE_REVIEW']:
    normalized=df[c].astype(str).str.lower()
    if not normalized.isin(['true','false','1','0']).all():raise ValueError(f'{c}: 알 수 없는 값')
    df[c]=normalized.isin(['true','1'])
numcols=['X_COORD','Y_COORD','ACTIVITY_RESIDUAL','RANK','FLOW','PEER_MEDIAN']
for c in numcols:df[c]=pd.to_numeric(df[c],errors='raise')
if not np.isfinite(df[numcols].to_numpy()).all():raise ValueError('좌표/점수/유동 값에 결측·무한대가 있습니다.')
if df.duplicated(['REGION','STD_YM','AGE','GRID_ID']).any():raise ValueError('그룹 내 격자 중복')
transform=Transformer.from_crs(5179,4326,always_xy=True)
df['LON'],df['LAT']=transform.transform(df.X_COORD.to_numpy(),df.Y_COORD.to_numpy())
if not (df.LON.between(124,132)&df.LAT.between(33,39)).all():raise ValueError('변환한 좌표 범위가 한국 밖입니다. 좌표계 확인')
df['DEFICIT']=(-df.ACTIVITY_RESIDUAL).clip(lower=0)
positive=df.loc[df.DEFICIT>0,'DEFICIT']
COLOR_CAP=float(positive.quantile(.99)) if len(positive) else 1.
if COLOR_CAP<=0:COLOR_CAP=1.
summary=df.groupby(['REGION','STD_YM','AGE'],as_index=False).agg(grids=('GRID_ID','size'),selected=('SELECTED','sum'),min_residual=('ACTIVITY_RESIDUAL','min'),max_residual=('ACTIVITY_RESIDUAL','max'))
summary.to_csv(OUTPUT/'map_group_summary.csv',index=False,encoding='utf-8-sig')
display(summary)
print(f'총 {len(df):,}개 격자-월-연령 기록 / {len(summary)}개 조합')
print('공통 색상 상한:',COLOR_CAP)

## 2. 50m 도형과 대시보드 데이터 구성
[추정] 제공된 중심좌표에서 ±25m 사각형을 재구성합니다. 50m 격자 원본 폴리곤을 제공받은 것은 아닙니다.
[확실] 월별 관측·분석 가능 격자가 서로 다릅니다. 빈 공간은 0점이 아니라 현재 결과표에 없는 공간입니다.

In [ ]:
grids=df[['GRID_ID','DONG','X_COORD','Y_COORD','LON','LAT']].drop_duplicates('GRID_ID').reset_index(drop=True)
check=df.groupby('GRID_ID')[['X_COORD','Y_COORD','DONG']].nunique()
if (check>1).any().any():raise ValueError('GRID_ID별 위치/동 정보 충돌')
grid_index=pd.Series(np.arange(len(grids)),index=grids.GRID_ID)
df['GI']=df.GRID_ID.map(grid_index).astype(int)
corners=[]
for dx,dy in [(-25,-25),(25,-25),(25,25),(-25,25)]:
    lon,lat=transform.transform(grids.X_COORD.to_numpy()+dx,grids.Y_COORD.to_numpy()+dy)
    corners.append(np.column_stack([lat,lon]).round(7))
# grid record: [id,dong,lat,lon,[[lat,lon],...]]
grid_data=[[r.GRID_ID,r.DONG,round(r.LAT,7),round(r.LON,7),[c[i].tolist() for c in corners]] for i,r in enumerate(grids.itertuples())]
groups={}
for (region,month,age),d in df.groupby(['REGION','STD_YM','AGE']):
    d=d.sort_values('RANK')
    # row: [grid index,residual,rank,selected,flow,peer median,peer review,percentile]
    n=len(d)
    pct=(100*(n-d.RANK)/(n-1)).clip(0,100) if n>1 else pd.Series([100.],index=d.index)
    rows=[]
    for r,p in zip(d.itertuples(),pct):
        rows.append([int(r.GI),round(r.ACTIVITY_RESIDUAL,6),int(r.RANK),int(r.SELECTED),round(r.FLOW,4),round(r.PEER_MEDIAN,4),int(r.PEER_DISTANCE_REVIEW),round(float(p),3)])
    groups['|'.join([region,month,age])]=rows
payload=dict(grids=grid_data,groups=groups,regions=sorted(df.REGION.unique()),months=sorted(df.STD_YM.unique()),ages=['20G','30G','40G','50G','60GU'],cap=COLOR_CAP,source=str(SOURCE))
assert sum(map(len,groups.values()))==len(df)
print('고유 격자:',len(grids),'그룹:',len(groups))

## 3. 대화형 지도 저장
[확실] 월·연령·지역·모드·점수 필터를 바꿀 수 있습니다. 격자를 클릭하면 원점수·관측 유동·비교집단 중앙값·선정 여부가 보입니다.
[추정] 히트맵은 강도를 합산해 부드럽게 보여줍니다. 격자 수·밀도에도 영향을 받으므로 수치 해석은 툴팁과 50m 격자 모드를 기준으로 하세요.

In [ ]:
template=r"""<!doctype html>
<html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>월별 활동 저조 공간 지도</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
<style>
:root{font-family:system-ui,"Malgun Gothic",sans-serif;color:#172033}*{box-sizing:border-box}body{margin:0;background:#f3f5f9}
#map{position:fixed;inset:0 0 0 330px}.panel{position:fixed;left:0;top:0;bottom:0;width:330px;overflow:auto;padding:24px;background:white;z-index:1000;box-shadow:3px 0 20px #15243a12}
h1{font-size:23px;line-height:1.35;margin:8px 0}p{font-size:12px;line-height:1.7;color:#657187}.eyebrow{font-size:11px;color:#2563eb;letter-spacing:1px;font-weight:700}
label{display:block;font-size:12px;font-weight:700;margin:17px 0 7px}select,button,input[type=number]{width:100%;border:1px solid #d7dfeb;border-radius:8px;padding:10px;background:white;color:#172033}
button{cursor:pointer;background:#172b4d;color:white;border:0;margin-top:12px}button.secondary{background:#edf2fa;color:#24436a}
.row{display:flex;gap:8px}.row>*{flex:1;min-width:0}.checks label{font-weight:400;margin:11px 0}
.stats{background:#f1f5fb;border-radius:10px;padding:13px;font-size:12px;line-height:1.9;margin-top:18px}.stats b{font-size:20px}
.legend{margin-top:18px;font-size:11px}.bar{height:12px;border-radius:6px;background:linear-gradient(90deg,#f8e8b3,#fcba58,#ef6a39,#ca262f,#750d37)}
.ends{display:flex;justify-content:space-between;margin-top:5px}.note{padding-top:15px;border-top:1px solid #edf0f5;margin-top:15px}
#status{position:fixed;right:14px;bottom:25px;z-index:1100;background:white;padding:9px 13px;border-radius:8px;font-size:12px;box-shadow:0 2px 15px #0002}
#loadError{display:none;color:#a40000;background:#fff0f0;padding:10px;font-size:12px}a{color:#2563eb}
@media(max-width:700px){.panel{width:260px;padding:16px}#map{left:260px}h1{font-size:19px}}
</style></head><body>
<aside class="panel"><span class="eyebrow">MONTHLY SPATIAL EXPLORER</span><h1>활동이 저조한 공간,<br>지도에서 살펴보기</h1>
<p>2025년 7–12월 · 강남구와 춘천시<br>월별·연령별 유사 환경 대비 활동 수준</p><div id="loadError">지도 라이브러리를 불러오지 못했습니다. 인터넷 연결 후 파일을 다시 여세요.</div>
<label>지역</label><select id="region"></select>
<div class="row"><div><label>월</label><select id="month"></select></div><div><label>연령</label><select id="age"></select></div></div>
<div class="row"><button class="secondary" id="prev">← 이전 월</button><button class="secondary" id="next">다음 월 →</button></div>
<label>표시 방식</label><select id="mode"><option value="grid">50m 격자 색상</option><option value="heat">부드러운 히트맵</option></select>
<label>색상의 기준</label><select id="metric"><option value="deficit">활동 저조 강도 · 전체 공통 색상</option><option value="percentile">활동 저조 상대순위 · 0~100</option></select>
<p id="metricNote"></p>
<label>원점수 범위 <span style="font-weight:400">(작을수록 활동 저조)</span></label>
<div class="row"><input id="min" type="number" step=".1" aria-label="원점수 최소"><input id="max" type="number" step=".1" aria-label="원점수 최대"></div>
<div class="checks"><label><input id="selected" type="checkbox" checked> 선정된 격자만 보기</label><label><input id="review" type="checkbox"> 비교집단 거리 검토 대상만 보기</label></div>
<label>표시 진하기 <span id="opacityValue">65%</span></label><input id="opacity" type="range" min="15" max="95" value="65" style="width:100%">
<button id="apply">조건 적용</button><button id="fit" class="secondary">표시 공간에 지도 맞추기</button>
<div class="stats" id="stats"></div><div class="legend"><strong id="legendTitle"></strong><div class="bar" style="margin-top:8px"></div><div class="ends"><span id="legendLow"></span><span id="legendHigh"></span></div></div>
<p class="note">회색 격자: 관측 유동이 비교집단 이상인 공간.<br>빈 공간: 현재 분석 결과에 없는 공간. 0점이 아닙니다.<br><b>사회적 고립 확정 지도가 아닙니다.</b><br>히트맵의 진하기는 점의 밀도·줌에도 영향을 받습니다. 월별 점수 비교는 격자 모드를 사용하세요.</p>
<p>공통 색상 상한은 전체 양수 강도 99백분위입니다. 상한을 넘는 원점수는 삭제하지 않습니다.</p>
</aside><div id="map"></div><div id="status">지도 준비 중…</div>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script src="https://unpkg.com/leaflet.heat@0.2.0/dist/leaflet-heat.js"></script>
<script>
const DATA=__PAYLOAD__;
const el=id=>document.getElementById(id),fmt=(n,d=2)=>Number(n).toLocaleString('ko-KR',{maximumFractionDigits:d});
const ageNames={'20G':'20대','30G':'30대','40G':'40대','50G':'50대','60GU':'60세 이상'};
function options(id,values,label){values.forEach(v=>{const o=document.createElement('option');o.value=v;o.textContent=label?label(v):v;el(id).appendChild(o)})}
options('region',DATA.regions);options('month',DATA.months,v=>v.slice(0,4)+'.'+v.slice(4));options('age',DATA.ages,v=>ageNames[v]);
el('region').value=DATA.regions.includes('강남구')?'강남구':DATA.regions[0];
const lo=Math.floor(Math.min(...Object.values(DATA.groups).map(g=>Math.min(...g.map(r=>r[1]))))*10)/10,hi=Math.ceil(Math.max(...Object.values(DATA.groups).map(g=>Math.max(...g.map(r=>r[1]))))*10)/10;
el('min').value=lo;el('max').value=hi;
let map,gridLayer,heatLayer,lastRows=[],lastBounds=null,fitNext=true;
const stops=[[248,232,179],[252,186,88],[239,106,57],[202,38,47],[117,13,55]];
function color(t){if(t<=0)return '#b3becb';t=Math.max(0,Math.min(1,t));const f=t*(stops.length-1),i=Math.min(stops.length-2,Math.floor(f)),a=f-i;return 'rgb('+stops[i].map((v,j)=>Math.round(v*(1-a)+stops[i+1][j]*a)).join(',')+')'}
function intensity(r){return el('metric').value==='percentile'?r[7]/100:Math.min(1,Math.max(0,-r[1])/DATA.cap)}
function esc(s){return String(s).replace(/[&<>"']/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]))}
function popup(r){const g=DATA.grids[r[0]];return '<b>'+esc(g[1])+' · '+ageNames[el('age').value]+'</b><br>'+el('month').value+' / '+esc(el('region').value)+'<hr>원점수: <b>'+fmt(r[1],6)+'</b><br>활동 저조 강도: '+fmt(Math.max(0,-r[1]),6)+'<br>활동 저조 상대순위: '+fmt(r[7],2)+'/100<br>순위: '+fmt(r[2],0)+'<br>관측 유동: '+fmt(r[4],4)+'<br>비교집단 중앙값: '+fmt(r[5],4)+'<br>선정: '+(r[3]?'예':'아니오')+'<br>비교집단 거리 검토: '+(r[6]?'필요':'아니오')+'<br>위도/경도: '+g[2]+', '+g[3]+'<br><small>'+esc(g[0])+'</small>'}
function updateLegend(){const relative=el('metric').value==='percentile';el('legendTitle').textContent=relative?'활동 저조 상대순위':'활동 저조 강도 · 모든 월/지역/연령에 같은 색상';el('legendLow').textContent='낮음 (0)';el('legendHigh').textContent=relative?'높음 (100)':'높음 ('+fmt(DATA.cap,3)+' 이상)';el('metricNote').textContent=relative?'100에 가까울수록 같은 지역·월·연령 안에서 원점수가 낮습니다. 그룹 간 절대 점수 비교용은 아닙니다.':'양수 강도=-원점수. 값이 클수록 유사 환경 대비 유동이 낮습니다. 같은 색은 같은 강도 범위입니다.'}
function draw(){if(!map)return;const key=[el('region').value,el('month').value,el('age').value].join('|'),base=DATA.groups[key]||[],min=Number(el('min').value),max=Number(el('max').value);
if(min>max){el('status').textContent='점수 최소값이 최대값보다 큽니다.';return}
lastRows=base.filter(r=>r[1]>=min&&r[1]<=max&&(!el('selected').checked||r[3])&&(!el('review').checked||r[6]));
gridLayer.clearLayers();if(heatLayer){map.removeLayer(heatLayer);heatLayer=null}
const opacity=Number(el('opacity').value)/100;
if(el('mode').value==='grid'){lastRows.forEach(r=>{const g=DATA.grids[r[0]],t=intensity(r);L.polygon(g[4],{stroke:false,fillColor:color(t),fillOpacity:opacity}).bindPopup(popup(r),{maxWidth:350}).addTo(gridLayer)})}
else if(typeof L.heatLayer==='function'){const pts=lastRows.filter(r=>intensity(r)>0).map(r=>{const g=DATA.grids[r[0]];return[g[2],g[3],intensity(r)]});heatLayer=L.heatLayer(pts,{radius:20,blur:18,maxZoom:18,max:1,minOpacity:.1,gradient:{0.1:'#f8e8b3',0.35:'#fcba58',0.6:'#ef6a39',0.8:'#ca262f',1:'#750d37'}}).addTo(map);heatLayer._canvas.style.opacity=opacity}
else{el('status').textContent='히트맵 라이브러리 로딩 실패. 격자 모드를 사용하세요.'}
lastBounds=lastRows.length?L.latLngBounds(lastRows.map(r=>[DATA.grids[r[0]][2],DATA.grids[r[0]][3]])):null;
if(fitNext&&lastBounds){map.fitBounds(lastBounds,{padding:[25,25],maxZoom:16});fitNext=false}
const shownSelected=lastRows.filter(r=>r[3]).length,med=lastRows.map(r=>r[1]).sort((a,b)=>a-b),median=med.length?(med[Math.floor((med.length-1)/2)]+med[Math.floor(med.length/2)])/2:null;
el('stats').innerHTML='표시 격자 <b>'+fmt(lastRows.length,0)+'</b>개<br>현재 그룹 전체 '+fmt(base.length,0)+'개 · 전체 선정 '+fmt(base.filter(r=>r[3]).length,0)+'개<br>필터 안 선정 '+fmt(shownSelected,0)+'개<br>표시 격자 원점수 중앙값 '+(median===null?'—':fmt(median,4));
el('status').textContent=el('month').value+' · '+el('region').value+' · '+ageNames[el('age').value]+' · '+fmt(lastRows.length,0)+'개';updateLegend();}
function init(){if(typeof L==='undefined'){el('loadError').style.display='block';el('status').textContent='인터넷 연결을 확인하세요';return}
map=L.map('map',{preferCanvas:true}).setView([37.5,127.05],12);const light=L.tileLayer('https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png',{attribution:'© OpenStreetMap © CARTO',maxZoom:20}).addTo(map),osm=L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png',{attribution:'© OpenStreetMap',maxZoom:19});
L.control.layers({'밝은 지도':light,'일반 지도':osm}).addTo(map);L.control.scale({imperial:false}).addTo(map);gridLayer=L.layerGroup().addTo(map);
['region','month','age','metric','mode','selected','review'].forEach(id=>el(id).onchange=()=>{if(id==='region')fitNext=true;draw()});
el('opacity').oninput=()=>{el('opacityValue').textContent=el('opacity').value+'%';draw()};el('apply').onclick=draw;el('fit').onclick=()=>{if(lastBounds)map.fitBounds(lastBounds,{padding:[25,25],maxZoom:16})};
['prev','next'].forEach(id=>el(id).onclick=()=>{const i=DATA.months.indexOf(el('month').value)+(id==='next'?1:-1);if(i>=0&&i<DATA.months.length){el('month').value=DATA.months[i];draw()}});
map.on('click',e=>{if(el('mode').value!=='heat'||!lastRows.length)return;let best=null,dist=Infinity;lastRows.forEach(r=>{const g=DATA.grids[r[0]],d=map.distance(e.latlng,[g[2],g[3]]);if(d<dist){dist=d;best=r}});if(best&&dist<150)L.popup().setLatLng([DATA.grids[best[0]][2],DATA.grids[best[0]][3]]).setContent(popup(best)).openOn(map)});
draw();}
init();
</script></body></html>"""
dashboard=template.replace('__PAYLOAD__',json.dumps(payload,ensure_ascii=False,separators=(',',':'),allow_nan=False).replace('</','<\\/'))
MAP_PATH=OUTPUT/'monthly_age_score_dashboard.html'
MAP_PATH.write_text(dashboard,encoding='utf-8')
meta=dict(source=str(SOURCE/'04_all_scores.csv'),rows=len(df),groups=len(groups),unique_grids=len(grids),score='ACTIVITY_RESIDUAL lower means lower activity',color_metric='max(0,-ACTIVITY_RESIDUAL)',common_color_cap=COLOR_CAP,color_cap_rule='99th percentile of all positive deficits',relative_percentile='100*(N-rank)/(N-1) within region/month/age',crs_source='EPSG:5179',crs_map='EPSG:4326',heatmap_note='sum of local weights, affected by density and zoom; use grid mode for score comparison')
(OUTPUT/'map_settings.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
print('지도 완성:',MAP_PATH)
print(f'HTML 크기: {MAP_PATH.stat().st_size/1024**2:.1f} MB')
display(HTML('<p><b>완료.</b> 아래 경로의 HTML을 파일 탐색기에서 더블클릭하세요.</p><code>'+html.escape(str(MAP_PATH))+'</code>'))
if OPEN_BROWSER:webbrowser.open(MAP_PATH.as_uri())

## 4. VS Code 안에서 미리 보기 (선택)
[확실] 아래 셀은 기본적으로 실행하지 않습니다. PREVIEW_IN_NOTEBOOK=True로 바꾸면 같은 대시보드를 노트북 안에 넣습니다. 대용량 HTML이므로 외부 브라우저에서 여는 편이 가볍습니다. 실행 후 노트북 저장 시 지도 데이터가 출력에 포함될 수 있습니다.

In [ ]:
PREVIEW_IN_NOTEBOOK=False
if PREVIEW_IN_NOTEBOOK:
    iframe='<iframe srcdoc="'+html.escape(dashboard,quote=True)+'" style="width:100%;height:850px;border:1px solid #ddd;border-radius:12px"></iframe>'
    display(HTML(iframe))

[확실] 생성 파일은 대시보드 HTML, 그룹별 요약 CSV, 지도 설정 JSON입니다. 입력 CSV와 선정 여부는 바꾸지 않습니다.

사용 순서: **지역 → 월 → 연령 → 색상 기준 → 필요하면 점수 범위 입력 후 조건 적용**.
기본은 선정 격자만 표시합니다. 전체 점수를 보려면 '선정된 격자만 보기'를 해제하세요.
[추정] 원점수는 그룹별 비교집단으로 계산된 값입니다. 공통 색상은 같은 숫자를 같은 색으로 표시하지만, 동일한 사회적 고립 확률을 보장하지 않습니다.